In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
!! export NUMBA_CACHE_DIR="/local/ritzert/numba_cache"
!! export NUMBA_DISABLE_CACHING=1

[]

In [2]:
import corc.graph_metrics.neb
import numpy as np
import corc.utils
import corc.visualization
import corc.tmm_plots
import corc.our_algorithms
import sklearn.metrics
import tqdm
import scipy.sparse
import anndata
import scanpy
import pynndescent

In [4]:
# dataset = "densired_circles_16"
dataset = "noisy_moons"
X,y,_, params = corc.utils.load_dataset(dataset, index=0, return_params=True)

In [5]:
counts = scipy.sparse.csr_matrix(X, dtype=np.float32)

In [6]:
adata = anndata.AnnData(counts)

In [7]:
scanpy.pp.neighbors(adata)

In [8]:
scanpy.tl.leiden(
            adata,
            flavor="igraph",
            n_iterations=2,
            resolution=0.1,
            random_state=42,
        )

In [9]:
y_pred = adata.obs["leiden"]
ari = sklearn.metrics.adjusted_rand_score(y,y_pred)
ari

0.27339309241118903

(10000,)

In [ ]:
leiden = corc.our_algorithms.get_clustering_objects(params, X=X, selector=["Leiden"])[0][1]

In [ ]:

print(type(adata.X), getattr(adata.X, "dtype", None))
print("has PCA:", "X_pca" in adata.obsm)


<class 'scipy.sparse._csr.csr_matrix'> float32
has PCA: False


In [ ]:
leiden.fit(X)

ValueError: `method` needs to be one of dict_keys(['umap', 'gauss']).

: 

In [ ]:
y_pred = corc.utils.get_prediction(leiden, X, len(np.unique(y)))

In [ ]:
sklearn.metrics.adjusted_rand_score(y,y_pred)

0.5540475176562601

In [ ]:
algos = list()
for i in tqdm.trange(10):
    X,y,_, params = corc.utils.load_dataset(dataset, index=0, return_params=True)
    leiden = corc.our_algorithms.get_clustering_objects(params, X=X, selector=["Leiden"])[0][1]
    leiden.fit(X)
    algos.append(leiden)
    y_pred = corc.utils.get_prediction(leiden, X, len(np.unique(y)))
    ari = sklearn.metrics.adjusted_rand_score(y,y_pred)